In [1]:
import pandas as pd 
import numpy as np 


In [34]:
df =pd.read_csv('Data/PS_20174392719_1491204439457_log.csv')

In [35]:
df.shape

(6362620, 11)

In [36]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [37]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [39]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Make a copy from the data from for cleaning 

In [40]:
df_clean = df.copy()
df_clean = df_clean[df_clean["amount"] > 0]
df_clean = df_clean.dropna()


`Observation:
The dataset contains no missing values. However, some transactions have non-positive amounts, which are invalid and will be removed during data cleaning.
`

In [41]:
df_clean.shape

(6362604, 11)

In [45]:
df_clean.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

 
- `step`
-Represents a simulated time step in hours. Each step corresponds to one hour of real time, starting from the beginning of the simulation.

- `type`
The type of transaction performed. Examples include TRANSFER, CASH_OUT, PAYMENT, and others. This column indicates the nature of the financial operation.

- `amount`
The monetary value of the transaction. It represents the amount of money transferred during the transaction.

- `nameOrig`
A unique identifier for the customer who initiated the transaction (the sender).

- `oldbalanceOrg`
The balance of the sender’s account before the transaction was executed.

- `newbalanceOrig`
The balance of the sender’s account after the transaction was completed.

- `nameDest`
A unique identifier for the recipient of the transaction (the destination account).

- `oldbalanceDest`
The balance of the recipient’s account before receiving the transaction amount.

- `newbalanceDest`
The balance of the recipient’s account after the transaction was processed.

- `isFraud`
A binary indicator showing whether the transaction is labeled as fraudulent. This column is used only for evaluation purposes and is not used in the risk scoring process.

- `isFlaggedFraud`
A binary indicator showing whether the transaction was flagged as fraudulent by the system based on predefined rules.`

`converts step values into durations`

In [113]:
base_time=pd.to_datetime("2003-12-10")
df_clean["datetime"]=base_time+pd.to_timedelta(df_clean["step"],unit="h")
df_clean["day"]=df_clean["datetime"].dt.day

In [114]:
df_clean

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,datetime,day
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0,2003-12-10 01:00:00,10
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0,2003-12-10 01:00:00,10
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0,2003-12-10 01:00:00,10
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0,2003-12-10 01:00:00,10
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0,2003-12-10 01:00:00,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0,2004-01-09 23:00:00,9
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0,2004-01-09 23:00:00,9
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0,2004-01-09 23:00:00,9
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0,2004-01-09 23:00:00,9


In [115]:
df_clean['type'].value_counts()

type
CASH_OUT    2237484
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

`fraud is appeared more in !CASH_IN`

`CASH_IN`

In [116]:
NON_CASH_IN=df_clean['type']!='CASH_IN'

In [117]:
2237484+2151495+532909+41432

4963320

In [118]:
print(NON_CASH_IN.sum())

4963320


In [119]:
new_balance=df_clean['oldbalanceOrg']-df_clean['amount']

In [120]:
diff_balance=abs(new_balance-df_clean['newbalanceOrig'])

In [121]:
invalid_balance=NON_CASH_IN & (diff_balance > 0.1)

In [122]:
print(invalid_balance.sum())

3601556


In [123]:
3601556/4963320

0.7256344543571641

`Statistical Features for the Customers`

In [124]:
customer_features=df_clean.groupby("nameOrig").agg(transaction_count=('amount','count'),
                                                   total_amount=('amount','sum'),
                                                   avg_amount=('amount','mean'),
                                                   max_amount=('amount','max'),
                                                   std_amount=('amount','std')).reset_index()

In [125]:
customer_features.shape

(6353291, 6)

In [126]:
customer_features


,nameOrig,transaction_count,total_amount,avg_amount,max_amount,std_amount
0,C1000000639,1,244486.46,244486.46,244486.46,NaN
1,C1000001337,1,3170.28,3170.28,3170.28,NaN
2,C1000001725,1,8424.74,8424.74,8424.74,NaN
3,C1000002591,1,261877.19,261877.19,261877.19,NaN
4,C1000003372,1,20528.65,20528.65,20528.65,NaN
...,...,...,...,...,...,...
6353286,C999996999,1,26585.43,26585.43,26585.43,NaN
6353287,C999998175,1,37516.21,37516.21,37516.21,NaN
6353288,C999999254,1,244962.21,244962.21,244962.21,NaN
6353289,C999999614,1,15714.59,15714.59,15714.59,NaN


In [127]:
customer_features['std_amount'].value_counts()
customer_features["std_amount"] = customer_features["std_amount"].fillna(0)


In [128]:
customer_features.isnull().sum()


nameOrig             0
transaction_count    0
total_amount         0
avg_amount           0
max_amount           0
std_amount           0
dtype: int64

`Time features`

In [129]:
df_clean['day']=df_clean['step']//24

In [130]:
df_clean.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,datetime,day
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,2003-12-10 01:00:00,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,2003-12-10 01:00:00,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,2003-12-10 01:00:00,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,2003-12-10 01:00:00,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,2003-12-10 01:00:00,0


In [138]:
daily_counts=(df_clean.groupby(['nameOrig','day']).size().reset_index(name="daily_tx_count"))

In [136]:
daily_counts

,nameOrig,day,dailt_tx_count
0,C1000000639,10,1
1,C1000001337,9,1
2,C1000001725,1,1
3,C1000002591,9,1
4,C1000003372,6,1
...,...,...,...
6362031,C999996999,15,1
6362032,C999998175,0,1
6362033,C999999254,1,1
6362034,C999999614,7,1


In [142]:
daily_velocity=(daily_counts.groupby('nameOrig')["daily_tx_count"].mean().reset_index(name="daily_velocity"))

In [144]:
customer_features=customer_features.merge(daily_velocity,on="nameOrig",how='left')

In [145]:
customer_features

,nameOrig,transaction_count,total_amount,avg_amount,max_amount,std_amount,daily_velocity
0,C1000000639,1,244486.46,244486.46,244486.46,0.0,1.0
1,C1000001337,1,3170.28,3170.28,3170.28,0.0,1.0
2,C1000001725,1,8424.74,8424.74,8424.74,0.0,1.0
3,C1000002591,1,261877.19,261877.19,261877.19,0.0,1.0
4,C1000003372,1,20528.65,20528.65,20528.65,0.0,1.0
...,...,...,...,...,...,...,...
6353286,C999996999,1,26585.43,26585.43,26585.43,0.0,1.0
6353287,C999998175,1,37516.21,37516.21,37516.21,0.0,1.0
6353288,C999999254,1,244962.21,244962.21,244962.21,0.0,1.0
6353289,C999999614,1,15714.59,15714.59,15714.59,0.0,1.0


In [146]:
customer_features['daily_velocity'].isnull().sum()

np.int64(0)

In [147]:
customer_features.columns

Index(['nameOrig', 'transaction_count', 'total_amount', 'avg_amount',
       'max_amount', 'std_amount', 'daily_velocity'],
      dtype='object')

In [148]:
df_clean["invalid_balance"] = (
    (df_clean["type"] != "CASH_IN") &
    (abs(df_clean["oldbalanceOrg"] - df_clean["amount"] - df_clean["newbalanceOrig"]) > 0.1)
)

invalid_balance_rate = (
    df_clean
    .groupby("nameOrig")["invalid_balance"]
    .mean()
    .reset_index(name="invalid_balance_rate")
)

customer_features = customer_features.merge(
    invalid_balance_rate, on="nameOrig", how="left"
)

customer_features["invalid_balance_rate"] = (
    customer_features["invalid_balance_rate"].fillna(0)
)


In [149]:
customer_features

,nameOrig,transaction_count,total_amount,avg_amount,max_amount,std_amount,daily_velocity,invalid_balance_rate
0,C1000000639,1,244486.46,244486.46,244486.46,0.0,1.0,1.0
1,C1000001337,1,3170.28,3170.28,3170.28,0.0,1.0,0.0
2,C1000001725,1,8424.74,8424.74,8424.74,0.0,1.0,1.0
3,C1000002591,1,261877.19,261877.19,261877.19,0.0,1.0,0.0
4,C1000003372,1,20528.65,20528.65,20528.65,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...
6353286,C999996999,1,26585.43,26585.43,26585.43,0.0,1.0,1.0
6353287,C999998175,1,37516.21,37516.21,37516.21,0.0,1.0,0.0
6353288,C999999254,1,244962.21,244962.21,244962.21,0.0,1.0,1.0
6353289,C999999614,1,15714.59,15714.59,15714.59,0.0,1.0,1.0


In [150]:
customer_features[
    ["transaction_count", "daily_velocity", "invalid_balance_rate"]
].describe()


,transaction_count,daily_velocity,invalid_balance_rate
count,6.353291e+06,6.353291e+06,6.353291e+06
mean,1.001466e+00,1.000089e+00,5.660497e-01
std,3.832007e-02,9.448623e-03,4.954373e-01
min,1.000000e+00,1.000000e+00,0.000000e+00
25%,1.000000e+00,1.000000e+00,0.000000e+00
50%,1.000000e+00,1.000000e+00,1.000000e+00
75%,1.000000e+00,1.000000e+00,1.000000e+00
max,3.000000e+00,2.000000e+00,1.000000e+00


In [155]:
df_clean.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud', 'datetime', 'day', 'invalid_balance'],
      dtype='object')

In [157]:
df_sorted = df_clean.sort_values(by=["nameOrig", "step"])


In [159]:
df_sorted['rolling_avg_amount']=(df_sorted.groupby("nameOrig")["amount"].rolling(window=10,min_periods=1).mean().reset_index(level=0,drop=True))

In [160]:
df_sorted['rolling_std_amount']=(df_sorted.groupby("nameOrig")["amount"].rolling(window=10,min_periods=1).std().reset_index(level=0,drop=True))

In [ ]:
rolling_features=df_sorted.groupby('nameOrig').agg(
    avg_rolling_amount=('rolling_avg_amount','mean'),
    max_rolling_amount=("rolling_avg_amount",'max')
).reset_index()

In [162]:
customer_features=customer_features.merge(rolling_features,on='nameOrig',how='left')

In [151]:
from scipy.stats import zscore

customer_features["amount_z"] = zscore(customer_features["total_amount"])
customer_features["velocity_z"] = zscore(customer_features["daily_velocity"])

customer_features["risk_score"] = (
    customer_features["amount_z"].abs() +
    customer_features["velocity_z"].abs() +
    customer_features["invalid_balance_rate"] * 2
)


In [163]:
def risk_band(score):
    if score < 2:
        return "Low"
    elif score < 4:
        return "Medium"
    elif score < 6:
        return "High"
    else:
        return "Critical"

customer_features["risk_level"] = customer_features["risk_score"].apply(risk_band)


In [173]:
customer_features.drop(columns='avg_rolling_std',inplace=True)

In [174]:
customer_features.isnull().sum()

nameOrig                0
transaction_count       0
total_amount            0
avg_amount              0
max_amount              0
std_amount              0
daily_velocity          0
invalid_balance_rate    0
amount_z                0
velocity_z              0
risk_score              0
risk_level              0
avg_rolling_amount      0
max_rolling_amount      0
dtype: int64